# Maia Corpus v9 — Gutenberg Acquisition Engine v0.6
**Aquisição integral por idioma · retomada · auditoria · Google Drive**

Fonte: `manu/project_gutenberg` no Hugging Face, splits `pt`, `en`, `es` (testados no Colab pelo projeto). **Não usa Gutendex.**

**Arquitetura:** o notebook Colab é o controlador; um worker isolado baixa por streaming e grava os livros **completos**, em arquivos individuais no Google Drive, sem truncamento. Um índice JSONL registra cada observação e um relatório JSON resume cada execução. Duplicatas por ID e SHA-256 são verificadas contra as aquisições anteriores **deste módulo v0.6**. A retomada reinicia o streaming e pula registros já persistidos; não há retomada por offset remoto. Nenhum livro é automaticamente liberado para treinamento.

**Direitos:** Gutenberg/HF não garante domínio público no Brasil; todo livro permanece em quarentena até revisão jurídica por obra/edição. Não incorpora ao corpus v8 nem faz tokenização.

## 1. Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, sys, subprocess, time, signal, datetime
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
BASE=ROOT/'gutenberg_v06'
for name in ['quarantine/books','manifests','reports','logs','workers','configs','state']:
    (BASE/name).mkdir(parents=True,exist_ok=True)
assert Path('/content/drive/MyDrive').exists()
print('Armazenamento persistente:',BASE)

## 2. Configuração da coleta

Comece com **10 livros únicos por idioma**; depois aumente `max_unique_per_language`. O limite se refere ao total de livros persistidos por idioma em todas as execuções v0.6, não a cada sessão. `max_scan_per_language` limita registros examinados para evitar loops longos. Não há truncamento do texto.

In [ ]:
CFG={
 'dataset':'manu/project_gutenberg',
 'languages':['pt','en','es'],
 'max_unique_per_language':10,
 'max_scan_per_language':300,
 'max_book_bytes':30_000_000,
 'source_timeout_seconds':600,
 'progress_every':10,
}
CONFIG=BASE/'configs'/'gutenberg_v06.json'
CONFIG.write_text(json.dumps(CFG,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(CFG,ensure_ascii=False,indent=2))

## 3. Ambiente isolado

Instala Hugging Face `datasets` no `virtualenv`, sem atualizar `requests` do Python principal do Colab. O ambiente temporário pode ser recriado após reinício; os dados continuam no Drive.

In [ ]:
VENV=Path('/content/maia-gutenberg-v06-env')
PY=VENV/'bin/python'
if not PY.exists():
    try:
        subprocess.run([sys.executable,'-m','virtualenv','--version'],check=True,capture_output=True,timeout=20)
    except (subprocess.CalledProcessError,subprocess.TimeoutExpired):
        subprocess.run([sys.executable,'-m','pip','install','-q','virtualenv'],check=True,timeout=180)
    subprocess.run([sys.executable,'-m','virtualenv',str(VENV)],check=True,timeout=120)
subprocess.run([str(PY),'-m','pip','install','-q','datasets>=3,<5','huggingface_hub>=0.30,<2','requests>=2.32,<3'],check=True,timeout=600)
subprocess.run([str(PY),'-c','import datasets; print("datasets:",datasets.__version__)'],check=True,timeout=30)

## 4. Worker de aquisição e auditoria

- Arquivos `.txt` em `quarantine/books/<idioma>/<sha256>.txt`, escritos atomicamente (`.part` → rename).
- Manifesto por idioma em `manifests/<idioma>.jsonl` com ID, hash, tamanho, fonte, idioma e motivo da quarentena, **sem texto**.
- SHA-256 do texto integral, deduplicação por **ID e conteúdo** entre execuções v0.6, por idioma.
- Arquivos antigos preservados. Se houver arquivo com hash válido mas sem manifesto (interrupção), o worker recupera o índice a partir do hash do arquivo, mas sem atribuir ID de obra.
- Limite de bytes por livro é uma **barreira de segurança**, não truncamento: livros acima do limite são rejeitados integralmente e registrados.
- O índice é auditável, mas não constitui aprovação legal ou verificação de qualidade.

In [ ]:
WORKER_SOURCE = "import argparse,datetime,hashlib,json,os,re,sys,time\nfrom pathlib import Path\nfrom datasets import load_dataset\n\ndef stamp(): return datetime.datetime.now(datetime.timezone.utc).isoformat()\ndef append_line(path,obj):\n path.parent.mkdir(parents=True,exist_ok=True)\n with path.open('a',encoding='utf-8') as f:\n  f.write(json.dumps(obj,ensure_ascii=False)+'\\n'); f.flush(); os.fsync(f.fileno())\ndef main():\n ap=argparse.ArgumentParser()\n for a in ('root','config','language'): ap.add_argument('--'+a,required=True)\n args=ap.parse_args(); cfg=json.loads(Path(args.config).read_text(encoding='utf-8'))\n lang=args.language; base=Path(args.root)/'gutenberg_v06'\n bookdir=base/'quarantine'/'books'/lang; bookdir.mkdir(parents=True,exist_ok=True)\n manifest=base/'manifests'/f'{lang}.jsonl'\n ids=set(); hashes=set(); invalid=0\n if manifest.exists():\n  for line in manifest.open(encoding='utf-8'):\n   try:\n    x=json.loads(line)\n    if x.get('status')=='saved':\n     if x.get('book_id'): ids.add(str(x['book_id']))\n     if x.get('sha256'): hashes.add(x['sha256'])\n   except (ValueError,TypeError): invalid+=1\n # Files may have been saved before manifest append due to interruption.\n orphan_files=0\n for path in bookdir.glob('*.txt'):\n  sha=path.stem\n  if re.fullmatch('[0-9a-f]{64}',sha) and sha not in hashes:\n   data=path.read_bytes()\n   if hashlib.sha256(data).hexdigest()==sha:\n    hashes.add(sha); orphan_files+=1\n    append_line(manifest,{'status':'saved','book_id':None,'sha256':sha,'language':lang,\n       'source':'recovered_orphan_file','bytes':len(data),'timestamp_utc':stamp(),\n       'review_status':'quarantine_legal_review_required'})\n run=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')\n report={'run':run,'language':lang,'scanned':0,'saved':0,'duplicate_id':0,'duplicate_sha':0,\n         'oversize':0,'missing_text':0,'invalid_rows':0,'existing_unique':len(hashes),\n         'orphan_files_recovered':orphan_files,'invalid_manifest_lines':invalid,'error':None}\n try:\n  print('OPEN',cfg['dataset'],'split',lang,flush=True)\n  ds=load_dataset(cfg['dataset'],split=lang,streaming=True,trust_remote_code=False)\n  print('READY',lang,'existing_unique=',len(hashes),flush=True)\n  for row in ds:\n   if len(hashes)>=int(cfg['max_unique_per_language']): break\n   if report['scanned']>=int(cfg['max_scan_per_language']): break\n   report['scanned']+=1\n   if not isinstance(row,dict): report['invalid_rows']+=1; continue\n   bid=str(row.get('id','')).strip()\n   txt=row.get('text')\n   if not isinstance(txt,str) or not txt: report['missing_text']+=1; continue\n   if bid and bid in ids: report['duplicate_id']+=1; continue\n   raw=txt.encode('utf-8'); size=len(raw)\n   if size>int(cfg['max_book_bytes']):\n    report['oversize']+=1\n    append_line(manifest,{'status':'skipped_oversize','book_id':bid,'bytes':size,\n                          'language':lang,'timestamp_utc':stamp()})\n    continue\n   sha=hashlib.sha256(raw).hexdigest()\n   if sha in hashes:\n    report['duplicate_sha']+=1\n    if bid: ids.add(bid)\n    append_line(manifest,{'status':'duplicate_sha','book_id':bid,'sha256':sha,\n                          'language':lang,'timestamp_utc':stamp()})\n    continue\n   final=bookdir/f'{sha}.txt'; tmp=bookdir/f'.{sha}.{os.getpid()}.part'\n   try:\n    with tmp.open('wb') as f: f.write(raw); f.flush(); os.fsync(f.fileno())\n    os.replace(tmp,final)\n   finally:\n    if tmp.exists(): tmp.unlink()\n   append_line(manifest,{'status':'saved','book_id':bid or None,'sha256':sha,'bytes':size,\n       'characters':len(txt),'language':lang,'source_dataset':cfg['dataset'],\n       'source_split':lang,'timestamp_utc':stamp(),\n       'review_status':'quarantine_legal_review_required','path':str(final)})\n   hashes.add(sha)\n   if bid: ids.add(bid)\n   report['saved']+=1\n   print('SAVED',lang,bid,sha[:12],size,'bytes',flush=True)\n   if report['scanned']%int(cfg['progress_every'])==0:\n    print('PROGRESS',json.dumps(report,ensure_ascii=False),flush=True)\n except Exception as e:\n  report['error']=f'{type(e).__name__}: {e}'\n  print('SOURCE ERROR',report['error'],flush=True)\n finally:\n  report['total_unique_after']=len(hashes)\n  report['completed_utc']=stamp()\n  rp=base/'reports'/f'gutenberg_v06_{lang}_{run}.json'\n  rp.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')\n  print('REPORT_PATH',rp,flush=True)\n  print('FINAL',json.dumps(report,ensure_ascii=False),flush=True)\nif __name__=='__main__': main()\n"
WORKER_PATH=BASE/'workers'/'gutenberg_worker_v06.py'
WORKER_PATH.write_text(WORKER_SOURCE,encoding='utf-8')
print(WORKER_PATH)

## 5. Executar por idioma — logs visíveis e timeout

O monitor interrompe **o grupo de processos** após 600 segundos por idioma. Os documentos já gravados continuam no Drive. Execute novamente para retomar, sem sobrescrever. Caso a fonte demore mais de 600 segundos por idioma, aumente o timeout conscientemente.

In [ ]:
def run_language(lang):
    logfile=BASE/'logs'/f'gutenberg_v06_{lang}_{datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")}.log'
    cmd=[str(PY),'-u',str(WORKER_PATH),'--root',str(ROOT),'--config',str(CONFIG),'--language',lang]
    print('\n===',lang,'===',flush=True)
    with logfile.open('w',encoding='utf-8') as log:
        proc=subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        start=time.monotonic(); pos=0; timed_out=False
        try:
            while proc.poll() is None:
                with logfile.open(encoding='utf-8',errors='replace') as reader:
                    reader.seek(pos); chunk=reader.read(); pos=reader.tell()
                if chunk: print(chunk,end='',flush=True)
                if time.monotonic()-start>CFG['source_timeout_seconds']:
                    timed_out=True; print('TIMEOUT',lang,flush=True)
                    os.killpg(proc.pid,signal.SIGTERM)
                    try: proc.wait(timeout=5)
                    except subprocess.TimeoutExpired:
                        os.killpg(proc.pid,signal.SIGKILL); proc.wait()
                    break
                time.sleep(1)
        except KeyboardInterrupt:
            if proc.poll() is None:
                os.killpg(proc.pid,signal.SIGTERM)
                try: proc.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    os.killpg(proc.pid,signal.SIGKILL); proc.wait()
            raise
        with logfile.open(encoding='utf-8',errors='replace') as reader:
            reader.seek(pos); print(reader.read(),end='',flush=True)
    return {'language':lang,'exit_code':proc.returncode,'timeout':timed_out,'log':str(logfile)}
RESULTS=[]
for language in CFG['languages']:
    RESULTS.append(run_language(language))
print(json.dumps(RESULTS,ensure_ascii=False,indent=2))

## 6. Auditoria de integridade — SHA-256 de cada arquivo

Valida que todos os arquivos indexados como `saved` existem, que seu conteúdo corresponde ao hash e que não foram truncados. Este teste pode demorar se você ampliar o piloto para milhares de livros.

In [ ]:
import hashlib
for lang in CFG['languages']:
    manifest=BASE/'manifests'/f'{lang}.jsonl'
    latest={}; invalid=0
    if not manifest.exists():
        print(lang,'sem manifesto'); continue
    for line in manifest.open(encoding='utf-8'):
        try:
            obj=json.loads(line)
            if obj.get('status')=='saved' and obj.get('sha256'):
                latest[obj['sha256']]=obj
        except ValueError: invalid+=1
    errors=[]
    for sha,obj in latest.items():
        path=BASE/'quarantine'/'books'/lang/f'{sha}.txt'
        if not path.exists(): errors.append(('missing',sha)); continue
        data=path.read_bytes()
        if hashlib.sha256(data).hexdigest()!=sha: errors.append(('hash_mismatch',sha))
        elif obj.get('bytes') is not None and len(data)!=obj['bytes']:
            errors.append(('size_mismatch',sha))
    print(lang,'livros únicos=',len(latest),'erros=',len(errors),'linhas inválidas=',invalid)
    if errors: print('Amostra de erros:',errors[:5])

## 7. Relatórios e próximo estágio

Verifique `gutenberg_v06/reports`, `manifests` e `quarantine/books`. A aquisição integral **não** implica aprovação jurídica. O próximo notebook deve avaliar direitos autorais por obra/edição/jurisdição, remover cabeçalhos Gutenberg, normalizar/deduplicar contra v8 e preparar amostras para o MaiaTokenizer-65k. **Não descarte os arquivos originais.**

In [ ]:
for path in sorted((BASE/'reports').glob('gutenberg_v06_*.json'))[-12:]:
    obj=json.loads(path.read_text(encoding='utf-8'))
    print(path.name,'scanned=',obj['scanned'],'saved=',obj['saved'],
          'dup_id=',obj['duplicate_id'],'dup_sha=',obj['duplicate_sha'],
          'error=',obj['error'])